# Joint intent + slot filling on MASSIVE en-US

A small transformer trained from scratch, following the team's [data loading guide](https://claude.ai/artifact/P7ZiwdxinFuBRcYL9Qtn3K) and [transformer guide](https://claude.ai/artifact/7rJ1hEvzUkMCgBjwB6bcX1).
A 3-layer encoder feeds two heads: attention pooling predicts the intent, and a per-token linear layer predicts the BIO tags. Only the GloVe and fastText word vectors are pretrained, and they stay frozen.

The notebook runs top to bottom after Kernel → Restart & Run All. Trained models are stored in `runs/`, so a re-run reuses them instead of training again.

## 1. Setup

Imports, device, and one `CONFIG` dict, so each ablation in section 9 is a one-line change. `CONFIG` is the guide's, plus `scaled` and `use_crf`, which runs T2 and T6 need.

In [ ]:
%pip install -q seqeval pytorch-crf

In [ ]:
import copy, glob, hashlib, json, os, random, re, tarfile, time, urllib.request, zipfile

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle
from seqeval.metrics import f1_score
from torch.utils.data import DataLoader
from torchcrf import CRF

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

DATA_DIR = "data"   # all downloads go here; git-ignored
RUNS_DIR = "runs"   # one checkpoint and one log per trained model; git-ignored
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

SEEDS = [0, 1, 2]
CONFIG = dict(d_model=256, n_heads=4, n_layers=3, d_ff=512, dropout=0.2, pos="relative", scaled=True,
              use_fasttext=False, use_char_cnn=True, use_crf=False,
              lr=5e-4, batch_size=32, epochs=100, patience=10, lam=0.5, word_dropout=0.05)

## 2. Data

Each step lists the numbers the data loading guide expects. They were measured on a version 1.0 copy of the file.

### Pre-processing 1: Download the data

Only one file from the official release is used: `en-US.jsonl`, with the split marked on each row.
To use version 1.0 instead, change the URL's ending to `1.0.tar.gz` and re-run the notebook.

**Expected:** `en-US.jsonl` found on disk.

In [ ]:
url = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz"
archive = os.path.join(DATA_DIR, os.path.basename(url))
if not os.path.exists(archive):
    urllib.request.urlretrieve(url, archive)
with tarfile.open(archive) as tar:
    tar.extractall(os.path.join(DATA_DIR, "massive"), filter="data")
DATA_PATH = glob.glob(os.path.join(DATA_DIR, "massive", "**", "en-US.jsonl"), recursive=True)[0]
print(DATA_PATH)

### Pre-processing 2: Read the file and split it

Train fits the models, dev tunes them and picks checkpoints, and test is used once at the end.

**Expected:** 11,514 / 2,033 / 2,974 rows.

In [ ]:
def load_massive(path):
    splits = {"train": [], "dev": [], "test": []}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                splits[row["partition"]].append(row)
    return splits

splits = load_massive(DATA_PATH)
print({k: len(v) for k, v in splits.items()})

### Pre-processing 3: Convert annotations into words and BIO tags

The models need one tag per word, so each `annot_utt` becomes a word list plus a matching BIO tag list.
`to_examples` also checks that the words match `utt.split()`, so input words and tags always line up.

**Expected:** all checks pass; 11,344 slot spans in train; about 32% of utterances have no slots.

In [ ]:
SLOT_RE = re.compile(r"\[(\S+) : ([^\]]+)\]")

def annot_to_bio(annot):
    tokens, tags, pos = [], [], 0
    for m in SLOT_RE.finditer(annot):
        for w in annot[pos:m.start()].split():        # words before the slot
            tokens.append(w); tags.append("O")
        words = m.group(2).split()                     # words inside the slot
        tokens += words
        tags += [f"B-{m.group(1)}"] + [f"I-{m.group(1)}"] * (len(words) - 1)
        pos = m.end()
    for w in annot[pos:].split():                      # words after the last slot
        tokens.append(w); tags.append("O")
    return tokens, tags

def to_examples(rows):
    out = []
    for row in rows:
        tokens, tags = annot_to_bio(row["annot_utt"])
        assert tokens == row["utt"].split(), row["id"]
        out.append({"id": row["id"], "tokens": tokens, "tags": tags,
                    "intent": row["intent"], "annot_utt": row["annot_utt"]})
    return out

examples = {split: to_examples(rows) for split, rows in splits.items()}
spans = sum(t.startswith("B-") for e in examples["train"] for t in e["tags"])
no_slot = sum(all(t == "O" for t in e["tags"]) for e in examples["train"])
print(f"train: {spans} slot spans, {100 * no_slot / len(examples['train']):.1f}% of utterances have no slots")

### Parsing sanity check

The automatic checks must all pass; the five printed examples are for a visual check that words and tags line up.

**Expected:**

```
train: 11514 utterances, 3755 with no slots, 0 invalid I- tags
dev: 2033 utterances, 646 with no slots, 0 invalid I- tags
test: 2974 utterances, 994 with no slots, 0 invalid I- tags
```

In [ ]:
def bio_to_annot(tokens, tags):
    """Rebuild 'wake me up at [time : five am]' from words and BIO tags."""
    out, i = [], 0
    while i < len(tokens):
        if tags[i] == "O":
            out.append(tokens[i]); i += 1; continue
        label, span = tags[i][2:], [tokens[i]]; i += 1
        while i < len(tokens) and tags[i] == f"I-{label}":
            span.append(tokens[i]); i += 1
        out.append(f"[{label} : {' '.join(span)}]")
    return " ".join(out)

# Automatic checks: every one must pass
for split, exs in examples.items():
    assert all(len(e["tokens"]) == len(e["tags"]) for e in exs), split
    assert all(bio_to_annot(e["tokens"], e["tags"]) == " ".join(e["annot_utt"].split()) for e in exs), split
    bad_i = sum(t.startswith("I-") and p[2:] != t[2:]
                for e in exs for p, t in zip(["O"] + e["tags"], e["tags"]))
    no_slot = sum(all(t == "O" for t in e["tags"]) for e in exs)
    print(f"{split}: {len(exs)} utterances, {no_slot} with no slots, {bad_i} invalid I- tags")

# Five random examples, words lined up with their tags
random.seed(42)
for i, ex in enumerate(random.sample(examples["train"], 5), 1):
    w = [max(len(a), len(b)) for a, b in zip(ex["tokens"], ex["tags"])]
    print(f"\n[{i}] id={ex['id']}  intent={ex['intent']}")
    print("    utt      :", " ".join(ex["tokens"]))
    print("    annot_utt:", ex["annot_utt"])
    print("    words    :", "  ".join(a.ljust(n) for a, n in zip(ex["tokens"], w)))
    print("    tags     :", "  ".join(b.ljust(n) for b, n in zip(ex["tags"], w)))

## 3. Vocabularies and embeddings

### Pre-processing 4 and 5: Build vocabularies and label maps

Every vocabulary is built from **train only**, so dev and test stay unseen. Each one is sorted, so ids are reproducible.
The tag map lists every `B-` and `I-` tag, including `I-` tags train never uses, because those still need an id.

**Expected:**
- 60 intents; 111 tags (`O` + B-/I- for 55 slot types), 107 of them seen in train.
- 5,304 words (5,302 + 2); 34 characters (32 + 2); `<PAD>` = 0 and `<UNK>` = 1.

In [ ]:
def build_vocabs(train):
    intents = sorted({e["intent"] for e in train})
    slot_types = sorted({t[2:] for e in train for t in e["tags"] if t != "O"})
    tags = ["O"] + [f"{prefix}-{slot}" for slot in slot_types for prefix in "BI"]
    words = sorted({w for e in train for w in e["tokens"]})
    chars = sorted({c for w in words for c in w})
    specials = ["<PAD>", "<UNK>"]
    return {"word": {w: i for i, w in enumerate(specials + words)},
            "char": {c: i for i, c in enumerate(specials + chars)},
            "intent": {name: i for i, name in enumerate(intents)},
            "tag": {t: i for i, t in enumerate(tags)}}

vocabs = build_vocabs(examples["train"])
word2id, char2id, intent2id, tag2id = vocabs["word"], vocabs["char"], vocabs["intent"], vocabs["tag"]
id2tag = {i: t for t, i in tag2id.items()}

seen_tags = {t for e in examples["train"] for t in e["tags"]}
print(f"{len(intent2id)} intents; {len(tag2id)} tags, {len(seen_tags)} seen in train")
print(f"{len(word2id)} words; {len(char2id)} characters; <PAD> = {word2id['<PAD>']}, <UNK> = {word2id['<UNK>']}")

### Pre-processing 6: Build the GloVe and fastText matrices

Both matrices follow `word2id`'s order, so row *i* is the vector for word id *i*. `build_matrix` scans each file once and keeps only the vocabulary's rows.
The first run downloads about 1.45 GB of zips and leaves about 3.3 GB of extracted vectors in `data/`. The zips are deleted after extraction.

**Expected:** each matrix 5,304 × 300 (1,591,200 numbers), `<PAD>` row all zeros, and:

```
GloVe: matrix (5304, 300), missing 184 of 5302 words (3.47%), e.g. ['<empty>', "altucher's", 'apahelp', 'apig', 'applebees', 'badpho']
fastText: matrix (5304, 300), missing 140 of 5302 words (2.64%), e.g. ['<empty>', 'apahelp', "aren't", 'badpho', 'bbcnews', 'beanos']
missing from both: 126
```

The guide measured these counts on the gensim-data copies of the same two files.

In [ ]:
downloads = {
    "glove.6B.300d.txt": "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip",
    "wiki-news-300d-1M-subword.vec": "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M-subword.vec.zip",
}
for filename, url in downloads.items():
    if not os.path.exists(os.path.join(DATA_DIR, filename)):
        zip_path = os.path.join(DATA_DIR, os.path.basename(url))
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(filename, DATA_DIR)
        os.remove(zip_path)

In [ ]:
def split_apostrophe(word):
    """amazon's -> ('amazon', "'s"); aren't -> ('are', "n't")"""
    i = len(word) - 3 if word.endswith("n't") else word.index("'")
    return word[:i], word[i:]

def find(word, table):
    """Exact match, then Capitalised and UPPER forms (fastText is case-sensitive)."""
    for w in (word, word.capitalize(), word.upper()):
        if w in table:
            return table[w]
    return None

def lookup(word, table):
    vec = find(word, table)
    if vec is None and "'" in word:          # average the pieces GloVe/fastText store separately
        pieces = [find(p, table) for p in split_apostrophe(word)]
        if all(p is not None for p in pieces):
            vec = np.mean(pieces, axis=0)
    return vec

def build_matrix(path, word2id, dim=300, seed=0):
    """len(word2id) x dim matrix in word2id's order. <PAD> = zeros, <UNK> and missing words = random.
    Returns (matrix, list of vocabulary words that were not found)."""
    wanted = set(word2id)
    for w in word2id:
        if "'" in w:
            wanted.update(split_apostrophe(w))
    table = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")          # rstrip(): fastText lines end with a space
            if len(parts) == dim + 1 and parts[0].lower() in wanted:   # also skips the header line
                table[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
    rng = np.random.default_rng(seed)
    scale = np.std(np.stack(list(table.values())))   # random vectors on the same scale as real ones
    matrix = np.zeros((len(word2id), dim), dtype=np.float32)
    missing = []
    for w, idx in word2id.items():
        if w == "<PAD>":
            continue
        vec = None if w == "<UNK>" else lookup(w, table)
        if vec is None:
            vec = rng.normal(0.0, scale, dim)
            if w != "<UNK>":
                missing.append(w)
        matrix[idx] = vec
    return matrix, missing

glove_matrix, glove_missing = build_matrix(os.path.join(DATA_DIR, "glove.6B.300d.txt"), word2id)
fasttext_matrix, fasttext_missing = build_matrix(os.path.join(DATA_DIR, "wiki-news-300d-1M-subword.vec"), word2id)
n = len(word2id) - 2
for name, m, miss in [("GloVe", glove_matrix, glove_missing), ("fastText", fasttext_matrix, fasttext_missing)]:
    print(f"{name}: matrix {m.shape}, missing {len(miss)} of {n} words ({100 * len(miss) / n:.2f}%), e.g. {miss[:6]}")
    assert not m[word2id["<PAD>"]].any()
print("missing from both:", len(set(glove_missing) & set(fasttext_missing)))

### Pre-processing 7: Encode every sentence

Unseen words and characters in dev and test map to `<UNK>` (id 1). Lists stay variable-length; padding happens in section 4.

**Expected:** same counts as pre-processing 2, with one tag id per word. The one-word utterance "complaint" (id 15621) encodes to:

```python
{'word_ids': [974], 'char_ids': [[10, 22, 20, 23, 19, 8, 16, 21, 27]], 'tag_ids': [0], 'intent_id': 51}
```

In [ ]:
def encode(ex, vocabs):
    word2id, char2id = vocabs["word"], vocabs["char"]
    return {"word_ids": [word2id.get(w, word2id["<UNK>"]) for w in ex["tokens"]],
            "char_ids": [[char2id.get(c, char2id["<UNK>"]) for c in w] for w in ex["tokens"]],
            "tag_ids": [vocabs["tag"][t] for t in ex["tags"]],
            "intent_id": vocabs["intent"][ex["intent"]]}

encoded = {split: [encode(ex, vocabs) for ex in exs] for split, exs in examples.items()}

for split, xs in encoded.items():
    assert all(len(x["word_ids"]) == len(x["char_ids"]) == len(x["tag_ids"]) for x in xs), split
print({k: len(v) for k, v in encoded.items()})
print(next(x for e, x in zip(examples["train"], encoded["train"]) if e["id"] == "15621"))

### Pre-processing sanity checks

If a number differs, check the dataset version and the file path first.

| Check | Expected |
| --- | --- |
| Utterances (train / dev / test) | 11,514 / 2,033 / 2,974 |
| Vocab sizes (word / char / intent / tag) | 5,304 / 34 / 60 / 111 |
| Embedding parameters at 300d | 1,591,200 |
| Words per train utterance | 6.92 on average, 35 at most |
| Dev / test utterances with an unseen word | 17.90% / 18.93% |

In [ ]:
print("utterances:", {k: len(v) for k, v in examples.items()})
print("vocab sizes:", {"word": len(word2id), "char": len(char2id), "intent": len(intent2id), "tag": len(tag2id)})
print("numbers per 300d embedding matrix:", len(word2id) * 300)
lens = [len(e["tokens"]) for e in examples["train"]]
print(f"train words per utterance: mean {np.mean(lens):.2f}, max {max(lens)}")
for split in ("dev", "test"):
    unseen = sum(any(w not in word2id for w in e["tokens"]) for e in examples[split])
    print(f"{split} utterances with an unseen word: {100 * unseen / len(examples[split]):.2f}%")

## 4. Batching

`collate` pads each batch to its longest sentence and builds one mask that every later layer uses: True = real token. Tags are padded with -100, never 0, because tag id 0 is `O` and padding must not count as a real `O`.

In [ ]:
PAD, UNK, IGNORE = 0, 1, -100

def collate(batch, max_chars=20):
    B = len(batch)
    T = max(len(x["word_ids"]) for x in batch)
    C = min(max_chars, max(len(c) for x in batch for c in x["char_ids"]))
    words = torch.full((B, T), PAD, dtype=torch.long)
    chars = torch.full((B, T, C), PAD, dtype=torch.long)
    tags = torch.full((B, T), IGNORE, dtype=torch.long)   # -100 = ignored by the loss
    intents = torch.tensor([x["intent_id"] for x in batch], dtype=torch.long)
    for i, x in enumerate(batch):
        n = len(x["word_ids"])
        words[i, :n] = torch.tensor(x["word_ids"])
        tags[i, :n] = torch.tensor(x["tag_ids"])
        for j, cs in enumerate(x["char_ids"]):
            cs = cs[:C]
            chars[i, j, :len(cs)] = torch.tensor(cs)
    mask = words != PAD                                    # True = real token
    return words, chars, tags, intents, mask

train_data, dev_data, test_data = encoded["train"], encoded["dev"], encoded["test"]
dev_loader = DataLoader(dev_data, batch_size=64, collate_fn=collate)

words, chars, tags, intents, mask = next(iter(dev_loader))
print("words", words.shape, "chars", chars.shape, "tags", tags.shape, "intents", intents.shape, "mask", mask.shape)

## 5. Model

### Step 1: Embedding layer

Each word becomes frozen GloVe (300-d), plus optional fastText (300-d) and char-CNN (50-d) vectors, concatenated and projected to `d_model`. The projection keeps the transformer the same size for every embedding variant.

In [ ]:
class CharCNN(nn.Module):
    def __init__(self, n_chars, char_dim=30, out_dim=50, width=3):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=PAD)
        self.conv = nn.Conv1d(char_dim, out_dim, width, padding=width // 2)

    def forward(self, chars):                              # [B, T, C]
        B, T, C = chars.shape
        flat = chars.view(B * T, C)
        x = self.emb(flat).transpose(1, 2)                 # [B*T, char_dim, C]
        x = F.relu(self.conv(x))                           # [B*T, out_dim, C]
        x = x.masked_fill((flat == PAD).unsqueeze(1), 0.0) # padded chars can't win the max
        return x.max(dim=2).values.view(B, T, -1)          # [B, T, out_dim]

class WordEmbedder(nn.Module):
    def __init__(self, glove, d_model, fasttext=None, n_chars=None, char_out=50,
                 freeze=True, dropout=0.2):
        super().__init__()
        self.glove = nn.Embedding.from_pretrained(
            torch.as_tensor(glove, dtype=torch.float), freeze=freeze, padding_idx=PAD)
        in_dim = self.glove.embedding_dim
        self.fasttext = None
        if fasttext is not None:
            self.fasttext = nn.Embedding.from_pretrained(
                torch.as_tensor(fasttext, dtype=torch.float), freeze=freeze, padding_idx=PAD)
            in_dim += self.fasttext.embedding_dim
        self.char_cnn = None
        if n_chars is not None:
            self.char_cnn = CharCNN(n_chars, out_dim=char_out)
            in_dim += char_out
        self.proj = nn.Linear(in_dim, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, words, chars):
        parts = [self.glove(words)]
        if self.fasttext is not None:
            parts.append(self.fasttext(words))
        if self.char_cnn is not None:
            parts.append(self.char_cnn(chars))
        return self.drop(self.proj(torch.cat(parts, dim=-1)))  # [B, T, d_model]

### Step 2, version A: Built-in encoder, absolute positions

Self-attention ignores word order, so positions must be injected. Both versions use pre-LN, which trains stably from scratch with a short warmup, and both take `(x, mask)` and return token states `[B, T, d_model]`.

In [ ]:
class AbsPosEncoder(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, d_ff, dropout, max_len=128):
        super().__init__()
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.layers = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        positions = torch.arange(x.size(1), device=x.device)
        x = x + self.pos(positions)
        # PyTorch convention: src_key_padding_mask is True where the token is PADDING
        x = self.layers(x, src_key_padding_mask=~mask)
        return self.norm(x)

### Step 2, version B: Custom encoder, relative positions

Each head learns one bias per relative distance j − i between query word i and key word j, clipped to ±16. Because −2 and +2 get separate biases, the layer knows direction, which matters for BIO tagging.
`scaled=False` drops the 1/√d_k scaling, as in TENER (Yan et al., 2019).

In [ ]:
class RelPosSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, max_rel=16, scaled=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.max_rel = max_rel
        self.rel_bias = nn.Embedding(2 * max_rel + 1, n_heads)
        self.scale = self.dk ** -0.5 if scaled else 1.0     # scaled=False = TENER-style sharper attention
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):                             # x [B,T,D], mask [B,T] True = real
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)  # each [B,h,T,dk]
        scores = (q @ k.transpose(-1, -2)) * self.scale     # [B,h,T,T]
        pos = torch.arange(T, device=x.device)
        rel = (pos[None, :] - pos[:, None]).clamp(-self.max_rel, self.max_rel) + self.max_rel
        scores = scores + self.rel_bias(rel).permute(2, 0, 1)   # + [h,T,T]
        scores = scores.masked_fill(~mask[:, None, None, :], float("-inf"))  # never attend to PAD keys
        attn = self.drop(scores.softmax(dim=-1))
        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.out(out)

class RelPosEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, **attn_kw):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = RelPosSelfAttention(d_model, n_heads, dropout=dropout, **attn_kw)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
                                nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):                             # pre-LN: normalise before each sub-layer
        x = x + self.drop(self.attn(self.ln1(x), mask))
        x = x + self.drop(self.ff(self.ln2(x)))
        return x

class RelPosEncoder(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, d_ff, dropout, **attn_kw):
        super().__init__()
        self.layers = nn.ModuleList(
            RelPosEncoderLayer(d_model, n_heads, d_ff, dropout, **attn_kw) for _ in range(n_layers))
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

### Step 3: Intent head and slot head

The intent head attention-pools all token states into one sentence vector; the slot head classifies every token. Both read the same encoder output, which is what makes the model joint.
With `use_crf=True` the model also holds a CRF over the slot logits (run T6). `make_model` builds a model from a config dict.

**Expected:** about 1.7–1.8M trainable parameters at `d_model` 256.

In [ ]:
class AttnPool(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, h, mask):                             # h [B,T,D]
        s = self.score(h).squeeze(-1).masked_fill(~mask, float("-inf"))
        a = s.softmax(dim=-1)                               # [B,T] attention weights (plot these!)
        return (a.unsqueeze(-1) * h).sum(dim=1), a

class JointTransformer(nn.Module):
    def __init__(self, glove, n_intents, n_tags, fasttext=None, n_chars=None,
                 d_model=256, n_heads=4, n_layers=3, d_ff=512, dropout=0.2,
                 pos="relative", use_crf=False, **attn_kw):
        super().__init__()
        self.embed = WordEmbedder(glove, d_model, fasttext=fasttext, n_chars=n_chars, dropout=dropout)
        if pos == "relative":
            self.encoder = RelPosEncoder(d_model, n_heads, n_layers, d_ff, dropout, **attn_kw)
        else:
            self.encoder = AbsPosEncoder(d_model, n_heads, n_layers, d_ff, dropout)
        self.pool = AttnPool(d_model)
        self.drop = nn.Dropout(dropout)
        self.intent_head = nn.Linear(d_model, n_intents)
        self.slot_head = nn.Linear(d_model, n_tags)
        self.crf = CRF(n_tags, batch_first=True) if use_crf else None

    def forward(self, words, chars, mask):
        h = self.encoder(self.embed(words, chars), mask)   # [B,T,D]
        pooled, _ = self.pool(h, mask)                     # [B,D]
        return self.intent_head(self.drop(pooled)), self.slot_head(self.drop(h))

glove, fasttext = glove_matrix, fasttext_matrix

def make_model(cfg):
    return JointTransformer(glove, len(intent2id), len(id2tag),
                            fasttext=fasttext if cfg["use_fasttext"] else None,
                            n_chars=len(char2id) if cfg["use_char_cnn"] else None,
                            d_model=cfg["d_model"], n_heads=cfg["n_heads"], n_layers=cfg["n_layers"],
                            d_ff=cfg["d_ff"], dropout=cfg["dropout"], pos=cfg["pos"],
                            use_crf=cfg["use_crf"], scaled=cfg["scaled"]).to(device)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"trainable parameters: {count_params(make_model(CONFIG)):,}")

## 6. Loss and training helpers

### Step 4: Joint loss

One loss trains both heads through the shared encoder: λ · CE(intent) + (1 − λ) · CE(slot). Label smoothing applies to the intent only. With a CRF, its negative log-likelihood replaces the slot cross-entropy.

In [ ]:
def joint_loss(intent_logits, slot_logits, intents, tags, lam=0.5, smoothing=0.1, crf=None, mask=None):
    l_int = F.cross_entropy(intent_logits, intents, label_smoothing=smoothing)
    if crf is None:
        l_slot = F.cross_entropy(slot_logits.reshape(-1, slot_logits.size(-1)), tags.reshape(-1),
                                 ignore_index=IGNORE)
    else:
        l_slot = -crf(slot_logits, tags.clamp(min=0), mask=mask, reduction="mean")
    return lam * l_int + (1 - lam) * l_slot, l_int.detach(), l_slot.detach()

### Step 5: Training recipe

AdamW with linear warmup then linear decay, gradient clipping and word dropout. `train_epoch` returns the two losses separately, because a total that falls while one task stalls is easy to miss.

In [ ]:
def make_optimizer(model, lr=5e-4, weight_decay=0.01, warmup_steps=400, total_steps=20000):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue                                      # frozen GloVe / fastText
        (no_decay if p.ndim < 2 or "norm" in name or "ln" in name else decay).append(p)
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": weight_decay},
                             {"params": no_decay, "weight_decay": 0.0}], lr=lr)

    def schedule(step):                                   # linear warmup, then linear decay to 0
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, schedule)

def train_epoch(model, loader, opt, sched, device, lam=0.5, word_dropout=0.05, clip=1.0, smoothing=0.1):
    model.train()
    total = total_int = total_slot = 0.0
    for words, chars, tags, intents, mask in loader:
        words, chars, tags, intents, mask = (t.to(device) for t in (words, chars, tags, intents, mask))
        if word_dropout > 0:                              # randomly hide words -> forces use of context
            hide = (torch.rand(words.shape, device=device) < word_dropout) & mask
            words = words.masked_fill(hide, UNK)
        il, sl = model(words, chars, mask)
        loss, l_int, l_slot = joint_loss(il, sl, intents, tags, lam=lam, smoothing=smoothing,
                                         crf=model.crf, mask=mask)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        sched.step()
        total += loss.item()
        total_int += l_int.item()
        total_slot += l_slot.item()
    return {"loss": total / len(loader), "intent_loss": total_int / len(loader), "slot_loss": total_slot / len(loader)}

## 7. Evaluation

### Step 6: Decoding and evaluation

Take the argmax for the intent and each token's tag, repair invalid BIO sequences, then report intent accuracy, span-level slot F1 and exact match.
The guide says a CRF's paths never need repair, but `pytorch-crf` learns transition scores rather than hard rules, so `repair_bio` runs on CRF paths too.

In [ ]:
def repair_bio(tags):
    """Turn an I-X that doesn't continue an X span into B-X."""
    out, prev = [], "O"
    for t in tags:
        if t.startswith("I-") and prev[2:] != t[2:]:
            t = "B-" + t[2:]
        out.append(t)
        prev = t
    return out

@torch.no_grad()
def evaluate(model, loader, id2tag, device):
    model.eval()
    n = n_intent = n_exact = 0
    gold_all, pred_all = [], []
    for words, chars, tags, intents, mask in loader:
        words, chars, tags, intents, mask = (t.to(device) for t in (words, chars, tags, intents, mask))
        il, sl = model(words, chars, mask)
        ip = il.argmax(-1)
        paths = model.crf.decode(sl, mask=mask) if model.crf is not None else sl.argmax(-1).tolist()
        for b in range(words.size(0)):
            L = int(mask[b].sum())
            gold = [id2tag[t] for t in tags[b, :L].tolist()]
            pred = repair_bio([id2tag[t] for t in paths[b][:L]])
            gold_all.append(gold)
            pred_all.append(pred)
            ok_intent = ip[b].item() == intents[b].item()
            n_intent += ok_intent
            n_exact += ok_intent and gold == pred
            n += 1
    return {"intent_acc": n_intent / n, "slot_f1": f1_score(gold_all, pred_all), "exact_match": n_exact / n}

## 8. Smoke test

Run the guide's debugging checklist before any full training run.

In [ ]:
# Inputs: every utterance has words (an empty one gives NaN attention) and fits the position table
for split, xs in encoded.items():
    lengths = [len(x["word_ids"]) for x in xs]
    assert min(lengths) >= 1 and max(lengths) <= 128, split
assert word2id["<PAD>"] == PAD and word2id["<UNK>"] == UNK

# Padding: tags are -100 exactly where the mask is False
assert (tags[~mask] == IGNORE).all() and (tags[mask] >= 0).all()

# Embeddings: row 0 is zeros, padding_idx is set, and the vectors are frozen
check = make_model({**CONFIG, "use_fasttext": True})
for emb in (check.embed.glove, check.embed.fasttext):
    assert not emb.weight[PAD].any() and emb.padding_idx == PAD and not emb.weight.requires_grad
print(f"trainable parameters: GloVe {count_params(make_model({**CONFIG, 'use_char_cnn': False})):,}, "
      f"+ char-CNN {count_params(make_model(CONFIG)):,}, + fastText {count_params(check):,}")

# Metric: repair_bio matches the guide's example, and slot F1 scores whole spans, not tokens
assert repair_bio(["I-time", "I-time", "O", "I-date"]) == ["B-time", "I-time", "O", "B-date"]
assert f1_score([["B-time", "I-time", "O"]], [["B-time", "I-time", "O"]]) == 1.0
assert f1_score([["B-time", "I-time", "O"]], [["B-time", "O", "O"]]) == 0.0

# Mask direction: adding padding must not change a sentence's outputs in either encoder
short = min(train_data, key=lambda x: len(x["word_ids"]))
long = max(train_data, key=lambda x: len(x["word_ids"]))
for pos in ("absolute", "relative"):
    m = make_model({**CONFIG, "pos": pos}).eval()
    with torch.no_grad():
        alone = collate([short])
        padded = collate([short, long])
        il_a, sl_a = m(alone[0].to(device), alone[1].to(device), alone[4].to(device))
        il_p, sl_p = m(padded[0].to(device), padded[1].to(device), padded[4].to(device))
    L = len(short["word_ids"])
    assert (il_a[0] - il_p[0]).abs().max() < 1e-3 and (sl_a[0, :L] - sl_p[0, :L]).abs().max() < 1e-3, pos
print("input, padding, embedding, metric and mask checks pass")

### Overfit 64 training utterances

If the loss doesn't approach 0 and all three metrics don't reach about 1.0, the bug is in the code, not the hyperparameters. The guide's companion file `joint_transformer.py` wasn't available, so this cell is written from the guide's description and uses real data.

Dropout, word dropout and label smoothing are all off here. With smoothing on, the intent loss can't go below about 0.72 for 60 classes, so the loss could never approach 0.

In [ ]:
def overfit_check(overrides, n_utts=64, epochs=150, batch_size=16):
    random.seed(0); np.random.seed(0); torch.manual_seed(0)
    cfg = {**CONFIG, "dropout": 0.0, **overrides}
    model = make_model(cfg)
    tiny = train_data[:n_utts]
    tiny_train = DataLoader(tiny, batch_size=batch_size, shuffle=True, collate_fn=collate)
    tiny_eval = DataLoader(tiny, batch_size=n_utts, collate_fn=collate)
    opt, sched = make_optimizer(model, lr=cfg["lr"], warmup_steps=len(tiny_train),
                                total_steps=epochs * len(tiny_train))
    for _ in range(epochs):
        stats = train_epoch(model, tiny_train, opt, sched, device, lam=cfg["lam"], word_dropout=0.0, smoothing=0.0)
    return stats, evaluate(model, tiny_eval, id2tag, device)

variants = {"version A, absolute": {"pos": "absolute"},
            "version B, relative": {"pos": "relative"},
            "version B, scaled=False": {"scaled": False},
            "version B + fastText": {"use_fasttext": True},
            "version B + CRF": {"use_crf": True}}
for name, overrides in variants.items():
    start = time.time()
    stats, scores = overfit_check(overrides)
    print(f"{name:26s} loss {stats['loss']:.4f}  intent {scores['intent_acc']:.3f}  slot F1 {scores['slot_f1']:.3f}  "
          f"exact {scores['exact_match']:.3f}  ({time.time() - start:.0f}s)", flush=True)
    assert min(scores.values()) > 0.95, name

## 9. Training run

### Phase 1: Compare configurations on validation only

Each run below changes one thing from the current configuration, over 3 seeds, with early stopping on dev exact match (patience 10). This cell never touches test.

**Keep rule, fixed before training:** a change is kept only if it beats the current configuration on dev exact match beyond the seed spread, meaning the new mean − std is above the current mean + std (`np.std` over the 3 seeds).

| Run | Change | Question it answers |
| --- | --- | --- |
| T0 | Version A: absolute positions, GloVe, 3 layers, `d_model` 256 | Does a plain transformer work here at all? |
| T1 | Relative positions (version B) | Does direction-aware attention help tagging? |
| T2 | `scaled=False` (version B only) | Does TENER's sharper attention help on MASSIVE? |
| T3a, T3b | `d_model` 128, then 2 layers | Is a smaller model better on about 11k utterances? |
| T4 | Add char-CNN | Do character features help rare slot values? |
| T5 | Add fastText | Does a second word embedding add anything? |
| T6a, T6b | Add CRF at λ 0.5, then at λ 0.7 | Does tag-transition modelling beat simple repair? The CRF loss is larger, so the guide says to re-tune λ with it. |
| T7a, T7b | λ 0.3, then λ 0.7 | Step 4 says to tune λ on dev exact match. |

Compared with the guide's `run_experiment`, this one names each run by a hash of its config, so ablations don't overwrite each other's checkpoints. It also reuses finished runs and returns the per-epoch history for the training curves.

In [ ]:
def run_experiment(seed, cfg):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = make_model(cfg)
    run_id = hashlib.md5(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:10] + f"_seed{seed}"
    ckpt, log = os.path.join(RUNS_DIR, run_id + ".pt"), os.path.join(RUNS_DIR, run_id + ".json")
    if os.path.exists(log):                               # already trained: reuse it
        model.load_state_dict(torch.load(ckpt, map_location=device))
        with open(log, encoding="utf-8") as f:
            saved = json.load(f)
        return model, saved["dev"], saved["history"]
    start = time.time()
    train_loader = DataLoader(train_data, batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate)
    opt, sched = make_optimizer(model, lr=cfg["lr"], warmup_steps=len(train_loader),
                                total_steps=cfg["epochs"] * len(train_loader))
    best, best_state, bad, history = -1.0, None, 0, []
    for epoch in range(cfg["epochs"]):
        stats = train_epoch(model, train_loader, opt, sched, device,
                            lam=cfg["lam"], word_dropout=cfg["word_dropout"])
        dev = evaluate(model, dev_loader, id2tag, device)
        history.append({**stats, **dev})
        if dev["exact_match"] > best:
            best, best_state, bad = dev["exact_match"], copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= cfg["patience"]:
                break
    model.load_state_dict(best_state)
    dev = evaluate(model, dev_loader, id2tag, device)
    torch.save(best_state, ckpt)
    with open(log, "w", encoding="utf-8") as f:           # written last: a log means the run finished
        json.dump({"config": cfg, "seed": seed, "dev": dev, "history": history,
                   "seconds": time.time() - start}, f)
    return model, dev, history

def pct(scores, key):
    vals = [100 * s[key] for s in scores]
    return f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"

def beats(new, current, key="exact_match"):
    new, current = [s[key] for s in new], [s[key] for s in current]
    return np.mean(new) - np.std(new) > np.mean(current) + np.std(current)

In [ ]:
EXPERIMENTS = [
    ("T0", "Version A: absolute positions, GloVe", {"pos": "absolute", "use_char_cnn": False}),
    ("T1", "Relative positions (version B)", {"pos": "relative"}),
    ("T2", "`scaled=False`", {"scaled": False}),
    ("T3a", "`d_model` 128", {"d_model": 128, "d_ff": 256}),
    ("T3b", "2 layers", {"n_layers": 2}),
    ("T4", "Add char-CNN", {"use_char_cnn": True}),
    ("T5", "Add fastText", {"use_fasttext": True}),
    ("T6a", "Add CRF, λ 0.5", {"use_crf": True}),
    ("T6b", "Add CRF, λ 0.7", {"use_crf": True, "lam": 0.7}),
    ("T7a", "λ 0.3", {"lam": 0.3}),
    ("T7b", "λ 0.7", {"lam": 0.7}),
]

current, current_scores = dict(CONFIG), None
table = ["| Run | Change | Trainable params | Intent acc (%) | Slot F1 (%) | Exact match (%) | Mean epochs | Kept |",
         "| --- | --- | ---: | ---: | ---: | ---: | ---: | --- |"]
for run, change_name, change in EXPERIMENTS:
    if run == "T2" and current["pos"] != "relative":      # scaled only exists in version B
        continue
    cfg = {**current, **change}
    scores, epochs = [], []
    for seed in SEEDS:
        model, dev, history = run_experiment(seed, cfg)
        scores.append(dev)
        epochs.append(len(history))
        print(f"{run} seed {seed}: exact {100 * dev['exact_match']:.2f}, {len(history)} epochs", flush=True)
    kept = current_scores is None or beats(scores, current_scores)
    table.append(f"| {run} | {change_name} | {count_params(model):,} | {pct(scores, 'intent_acc')} | {pct(scores, 'slot_f1')} | "
                 f"{pct(scores, 'exact_match')} | {np.mean(epochs):.0f} | {'yes' if kept else 'no'} |")
    if kept:
        current, current_scores = cfg, scores
    del model

final_config = current
display(Markdown("\n".join(table)))
print("final config:", final_config)

## 10. Final test evaluation

### Step 7, phase 2: One final measurement

`final_config` is now fixed by the keep rule, and these are the same 3 seeds as its row in section 9, so their stored models are reused. Test is evaluated once per seed, and no decision may follow from these numbers. Validation is usually slightly higher, because it was used for selection.

In [ ]:
test_loader = DataLoader(test_data, batch_size=64, collate_fn=collate)
dev_scores, test_scores = [], []
for seed in SEEDS:
    model, dev, _ = run_experiment(seed, final_config)
    dev_scores.append(dev)
    test_scores.append(evaluate(model, test_loader, id2tag, device))

for split, scores in [("validation", dev_scores), ("test", test_scores)]:
    summary = {k: f"{np.mean([s[k] for s in scores]):.4f} ± {np.std([s[k] for s in scores]):.4f}"
               for k in ["intent_acc", "slot_f1", "exact_match"]}
    print(split, summary)

## 11. Results

### Comparison table

Test scores are percentages, mean ± std over 3 seeds. The BiLSTM baseline isn't in this notebook yet, so its row is empty. It must use the same splits, seeds and `evaluate` for the comparison to be fair.

In [ ]:
display(Markdown("\n".join([
    "| Model | Trainable params | Intent acc (%) | Slot F1 (%) | Exact match (%) |",
    "| --- | ---: | ---: | ---: | ---: |",
    "| BiLSTM baseline | | | | |",
    f"| Transformer, best config | {count_params(make_model(final_config)):,} | {pct(test_scores, 'intent_acc')} | "
    f"{pct(test_scores, 'slot_f1')} | {pct(test_scores, 'exact_match')} |",
])))

In [ ]:
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "figure.dpi": 130,
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans"], "font.size": 10,
    "text.color": INK_2, "axes.labelcolor": INK_2, "axes.titlecolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "semibold", "axes.titlelocation": "left", "axes.edgecolor": AXIS,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.7, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.major.size": 0, "ytick.major.size": 0, "legend.frameon": False,
})

### Training curves

The final configuration's three seeds, with each dot marking the checkpoint early stopping kept for that seed.

In [ ]:
histories = [run_experiment(seed, final_config)[2] for seed in SEEDS]
last = max(len(h) for h in histories)

fig, (ax_loss, ax_dev) = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
panels = [(ax_loss, "Training loss", "Mean loss per batch", 1, [("intent_loss", "Intent"), ("slot_loss", "Slot")]),
          (ax_dev, "Dev scores", "Score (%)", 100, [("intent_acc", "Intent accuracy"), ("slot_f1", "Slot F1"),
                                                     ("exact_match", "Exact match")])]
for ax, title, ylabel, scale, series in panels:
    for color, (key, label) in zip(SERIES, series):
        for i, history in enumerate(histories):
            ax.plot(range(1, len(history) + 1), [scale * e[key] for e in history], color=color, linewidth=1.5,
                    alpha=0.85, label=label if i == 0 else None)
        longest = max(histories, key=len)
        ax.annotate(label, (len(longest), scale * longest[-1][key]), xytext=(6, 0), textcoords="offset points",
                    va="center", color=INK_2)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_xlim(1, last * 1.22)
    ax.grid(axis="x", visible=False)
for i, history in enumerate(histories):
    best = int(np.argmax([e["exact_match"] for e in history]))
    ax_dev.scatter(best + 1, 100 * history[best]["exact_match"], s=45, color=SERIES[2], edgecolor=SURFACE,
                   linewidth=1.5, zorder=3, label="Kept checkpoint" if i == 0 else None)
ax_loss.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
ax_dev.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=4)
fig.tight_layout()
plt.show()

### Intent attention heat map

The attention-pooling weights show which words drove the intent decision. These are dev utterances from the seed 0 model: for each of six fixed intents, the first with 5 to 9 words. Each row's weights sum to 1.

In [ ]:
@torch.no_grad()
def intent_attention(model, x):
    words, chars, _, _, mask = (t.to(device) for t in collate([x]))
    model.eval()
    pooled, a = model.pool(model.encoder(model.embed(words, chars), mask), mask)
    return a[0].cpu().numpy(), model.intent_head(pooled).argmax(-1).item()

id2intent = {i: name for name, i in intent2id.items()}
attn_model = run_experiment(SEEDS[0], final_config)[0]
shown = []
for intent in ["alarm_set", "weather_query", "play_music", "calendar_set", "news_query", "iot_hue_lightoff"]:
    shown.append(next((e, x) for e, x in zip(examples["dev"], dev_data)
                      if e["intent"] == intent and 5 <= len(e["tokens"]) <= 9))
weights = [intent_attention(attn_model, x) for _, x in shown]

cmap = LinearSegmentedColormap.from_list("blues", BLUES)
norm = Normalize(0, max(a.max() for a, _ in weights))
fig, ax = plt.subplots(figsize=(11, 0.82 * len(shown) + 0.7))
for r, ((e, _), (a, pred)) in enumerate(zip(shown, weights)):
    y = len(shown) - 1 - r
    ax.text(0, y + 0.76, f"{e['intent']}  ·  predicted {id2intent[pred]}", fontsize=9, color=INK_2, va="bottom")
    for j, (word, weight) in enumerate(zip(e["tokens"], a)):
        ax.add_patch(Rectangle((j + 0.02, y), 0.96, 0.7, facecolor=cmap(norm(weight)), edgecolor="none"))
        ax.text(j + 0.5, y + 0.35, word, ha="center", va="center", fontsize=9,
                color="#ffffff" if norm(weight) > 0.45 else INK)
ax.set_xlim(0, 9)
ax.set_ylim(0, len(shown) + 0.05)
ax.axis("off")
ax.set_title("Attention weight per word")
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, fraction=0.018, pad=0.01, label="Attention weight").outline.set_visible(False)
plt.show()